In [1]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()  # repository root
(ROOT / "results" / "figures").mkdir(parents=True, exist_ok=True)
# ============================================================
# STEP 1: LOAD AND INSPECT DATASET NO. 2
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder

RANDOM_STATE = 42

# Load dataset
data = pd.read_csv(ROOT / "data" / "Dataset No.2.csv")

print("Dataset shape:", data.shape)
print("\nColumns:")
print(data.columns.tolist())

print("\nFirst 5 rows:")
display(data.head())

print("\nMissing values:")
missing = data.isnull().sum()
print(missing[missing > 0])

print("\nOriginal Behavior distribution:")
print(data["Behavior"].value_counts())

print("\nOriginal Behavior proportions:")
print((data["Behavior"].value_counts(normalize=True) * 100).round(2))

print("\nNumber of exact duplicate rows:")
print(data.duplicated().sum())

print("\nUnique ASTM materials:")
print(data["ASTM Material"].nunique())

Dataset shape: (1544, 21)

Columns:
['ASTM Material', 'C', 'Mn', 'Si', 'P', 'S', 'Cr', 'Mo', 'Ni', 'N', 'Ti', 'Nb ', 'Al', 'Fe balance (%)', 'Temperature', 'Behavior', 'Pitting_label', '[H+] (mol/dm3)', '[OH-] (mol/dm3)', '[Cl-] (mol/dm3)', '[S2-] (mol/dm3)']

First 5 rows:


,ASTM Material,C,Mn,Si,P,S,Cr,Mo,Ni,N,...,Nb,Al,Fe balance (%),Temperature,Behavior,Pitting_label,[H+] (mol/dm3),[OH-] (mol/dm3),[Cl-] (mol/dm3),[S2-] (mol/dm3)
0,403,0.15,1.00,0.5,0.040,0.03,11.5,0.0,0.60,0.0,...,0.0,0.0,86.180,20,Poor,0,2.6504,0.0,0.0,0.0
1,405,0.08,1.00,1.0,0.040,0.03,11.5,0.0,0.60,0.0,...,0.0,0.1,85.650,20,Poor,0,2.6504,0.0,0.0,0.0
2,S41050,0.04,1.00,1.0,0.045,0.03,10.5,0.0,0.60,0.1,...,0.0,0.0,86.685,20,Poor,0,2.6504,0.0,0.0,0.0
3,416,0.15,1.25,1.0,0.060,0.35,12.0,0.0,0.00,0.0,...,0.0,0.0,85.190,20,Poor,0,2.6504,0.0,0.0,0.0
4,410,0.15,1.00,1.0,0.040,0.03,11.5,0.0,0.75,0.0,...,0.0,0.0,85.530,20,Poor,0,2.6504,0.0,0.0,0.0



Missing values:
Series([], dtype: int64)

Original Behavior distribution:
Behavior
Resistant    839
Good         311
Poor         226
Severe       168
Name: count, dtype: int64

Original Behavior proportions:
Behavior
Resistant    54.34
Good         20.14
Poor         14.64
Severe       10.88
Name: proportion, dtype: float64

Number of exact duplicate rows:
0

Unique ASTM materials:
34


In [2]:
# ============================================================
# STEP 2: PREPARE FEATURES AND CREATE LOCKED TEST SET
# Dataset No. 2
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

RANDOM_STATE = 42
TEST_SIZE = 0.20

# ------------------------------------------------------------
# 1. Make a clean working copy
# ------------------------------------------------------------

df = data.copy()

# Clean accidental spaces in column names
df.columns = df.columns.str.strip()


# ------------------------------------------------------------
# 2. Encode target
# ------------------------------------------------------------

label_encoder = LabelEncoder()

y = pd.Series(
    label_encoder.fit_transform(df["Behavior"]),
    index=df.index,
    name="Behavior"
)

print("Class encoding:")
for i, label in enumerate(label_encoder.classes_):
    print(f"  {i} = {label}")


# ------------------------------------------------------------
# 3. Prepare features
# ------------------------------------------------------------

# Keep ASTM Material separately for later group-based validation
groups = df["ASTM Material"].copy()

# Remove identifier, target and alternative target
X = df.drop(
    columns=["ASTM Material", "Behavior", "Pitting_label"]
).copy()


# Remove zero-variance features
constant_features = [
    col for col in X.columns
    if X[col].nunique() <= 1
]


X = X.drop(columns=constant_features)

# ------------------------------------------------------------
# 3a. Clean feature names for XGBoost compatibility
# ------------------------------------------------------------

# XGBoost does not accept [, ] or < in feature names.
# This changes ONLY the column names, not the numerical data.

X.columns = (
    X.columns
    .str.strip()
    .str.replace("[", "", regex=False)
    .str.replace("]", "", regex=False)
    .str.replace("<", "", regex=False)
)

print("\nInitial number of candidate features:", X.shape[1])
print("Constant features removed:", constant_features)
print("Number of remaining candidate features:", X.shape[1])

print("\nFeature names:")
for i, feature in enumerate(X.columns, start=1):
    print(f"{i:2d}. {feature}")


# Make sure every predictor is numeric
non_numeric = X.select_dtypes(exclude=np.number).columns.tolist()

print("\nNon-numeric feature columns:", non_numeric)

if non_numeric:
    raise ValueError(
        f"Unexpected non-numeric features found: {non_numeric}"
    )


# ------------------------------------------------------------
# 4. Stratified 80:20 train-test split
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)


# ------------------------------------------------------------
# 5. Sanity checks
# ------------------------------------------------------------

print("\nDataset sizes:")
print(f"Full dataset : {len(X)}")
print(f"Training set : {len(X_train)}")
print(f"Test set     : {len(X_test)}")


print("\nTraining class distribution:")

train_distribution = pd.DataFrame({
    "Count": y_train.value_counts().sort_index(),
    "Percentage": (
        y_train.value_counts(normalize=True)
        .sort_index() * 100
    ).round(2)
})

train_distribution.index = [
    label_encoder.classes_[i]
    for i in train_distribution.index
]

display(train_distribution)


print("\nTest class distribution:")

test_distribution = pd.DataFrame({
    "Count": y_test.value_counts().sort_index(),
    "Percentage": (
        y_test.value_counts(normalize=True)
        .sort_index() * 100
    ).round(2)
})

test_distribution.index = [
    label_encoder.classes_[i]
    for i in test_distribution.index
]

display(test_distribution)


# ------------------------------------------------------------
# 6. Verify that train and test rows do not overlap
# ------------------------------------------------------------

overlap = X_train.index.intersection(X_test.index)

print("\nTrain-test row overlap:", len(overlap))

assert len(overlap) == 0
assert len(X_train) + len(X_test) == len(X)

print("\nLocked train-test split created successfully.")

Class encoding:
  0 = Good
  1 = Poor
  2 = Resistant
  3 = Severe

Initial number of candidate features: 17
Constant features removed: ['Temperature']
Number of remaining candidate features: 17

Feature names:
 1. C
 2. Mn
 3. Si
 4. P
 5. S
 6. Cr
 7. Mo
 8. Ni
 9. N
10. Ti
11. Nb
12. Al
13. Fe balance (%)
14. H+ (mol/dm3)
15. OH- (mol/dm3)
16. Cl- (mol/dm3)
17. S2- (mol/dm3)

Non-numeric feature columns: []

Dataset sizes:
Full dataset : 1544
Training set : 1235
Test set     : 309

Training class distribution:


,Count,Percentage
Good,249,20.16
Poor,181,14.66
Resistant,671,54.33
Severe,134,10.85



Test class distribution:


,Count,Percentage
Good,62,20.06
Poor,45,14.56
Resistant,168,54.37
Severe,34,11.00



Train-test row overlap: 0

Locked train-test split created successfully.


In [3]:
# ============================================================
# STEP 3: CLEAN BASELINE MODELS
# Dataset No. 2
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from xgboost import XGBClassifier


# ------------------------------------------------------------
# 1. Define baseline models
# ------------------------------------------------------------

baseline_models = {

    "Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "Bagging": BaggingClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        eval_metric="mlogloss",
        n_jobs=-1
    )
}


# ------------------------------------------------------------
# 2. Train and evaluate each baseline model
# ------------------------------------------------------------

baseline_results = []
fitted_baseline_models = {}

for name, model in baseline_models.items():

    print("=" * 70)
    print(f"Training: {name}")
    print("=" * 70)

    model = clone(model)

    start = time.time()

    # Train ONLY on training data
    model.fit(X_train, y_train)

    runtime = time.time() - start

    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Predicted probabilities for multiclass ROC-AUC
    y_test_proba = model.predict_proba(X_test)

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)

    gap = train_acc - test_acc

    macro_precision = precision_score(
        y_test,
        y_test_pred,
        average="macro",
        zero_division=0
    )

    macro_recall = recall_score(
        y_test,
        y_test_pred,
        average="macro",
        zero_division=0
    )

    macro_f1 = f1_score(
        y_test,
        y_test_pred,
        average="macro",
        zero_division=0
    )

    macro_auc = roc_auc_score(
        y_test,
        y_test_proba,
        multi_class="ovr",
        average="macro"
    )

    # Save fitted model
    fitted_baseline_models[name] = model

    # Save results
    baseline_results.append({
        "Model": name,
        "Train Accuracy": train_acc,
        "Test Accuracy": test_acc,
        "Gap": gap,
        "Macro Precision": macro_precision,
        "Macro Recall": macro_recall,
        "Macro F1": macro_f1,
        "Macro ROC-AUC": macro_auc,
        "Runtime (s)": runtime
    })

    print(f"Train Accuracy : {train_acc:.4f}")
    print(f"Test Accuracy  : {test_acc:.4f}")
    print(f"Train-Test Gap : {gap:.4f}")
    print(f"Macro Precision: {macro_precision:.4f}")
    print(f"Macro Recall   : {macro_recall:.4f}")
    print(f"Macro F1       : {macro_f1:.4f}")
    print(f"Macro ROC-AUC  : {macro_auc:.4f}")
    print(f"Runtime        : {runtime:.2f} s")
    print()


# ------------------------------------------------------------
# 3. Create comparison table
# ------------------------------------------------------------

baseline_results_df = pd.DataFrame(baseline_results)

baseline_results_df = baseline_results_df.sort_values(
    by="Test Accuracy",
    ascending=False
).reset_index(drop=True)


# Percentage formatting for easier reading
display_df = baseline_results_df.copy()

percentage_columns = [
    "Train Accuracy",
    "Test Accuracy",
    "Gap",
    "Macro Precision",
    "Macro Recall",
    "Macro F1",
    "Macro ROC-AUC"
]

for col in percentage_columns:
    display_df[col] = (display_df[col] * 100).round(2)

display_df["Runtime (s)"] = display_df["Runtime (s)"].round(2)


print("\n" + "=" * 90)
print("BASELINE MODEL COMPARISON - DATASET NO. 2")
print("=" * 90)

display(display_df)

Training: Decision Tree
Train Accuracy : 1.0000
Test Accuracy  : 0.9029
Train-Test Gap : 0.0971
Macro Precision: 0.8828
Macro Recall   : 0.8851
Macro F1       : 0.8834
Macro ROC-AUC  : 0.9251
Runtime        : 0.02 s

Training: Random Forest
Train Accuracy : 1.0000
Test Accuracy  : 0.7573
Train-Test Gap : 0.2427
Macro Precision: 0.7295
Macro Recall   : 0.7182
Macro F1       : 0.7222
Macro ROC-AUC  : 0.9341
Runtime        : 0.65 s

Training: Bagging
Train Accuracy : 1.0000
Test Accuracy  : 0.9320
Train-Test Gap : 0.0680
Macro Precision: 0.9237
Macro Recall   : 0.9176
Macro F1       : 0.9200
Macro ROC-AUC  : 0.9901
Runtime        : 21.44 s

Training: XGBoost
Train Accuracy : 1.0000
Test Accuracy  : 0.9450
Train-Test Gap : 0.0550
Macro Precision: 0.9451
Macro Recall   : 0.9236
Macro F1       : 0.9326
Macro ROC-AUC  : 0.9948
Runtime        : 23.07 s


BASELINE MODEL COMPARISON - DATASET NO. 2


,Model,Train Accuracy,Test Accuracy,Gap,Macro Precision,Macro Recall,Macro F1,Macro ROC-AUC,Runtime (s)
0,XGBoost,100.0,94.50,5.50,94.51,92.36,93.26,99.48,23.07
1,Bagging,100.0,93.20,6.80,92.37,91.76,92.00,99.01,21.44
2,Decision Tree,100.0,90.29,9.71,88.28,88.51,88.34,92.51,0.02
3,Random Forest,100.0,75.73,24.27,72.95,71.82,72.22,93.41,0.65


In [4]:
# ============================================================
# STEP 4: CLASS-SPECIFIC BASELINE PERFORMANCE
# ============================================================

from sklearn.metrics import classification_report, confusion_matrix

class_names = list(label_encoder.classes_)

for name, model in fitted_baseline_models.items():

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    y_pred = model.predict(X_test)

    print("\nClassification Report:")

    print(
        classification_report(
            y_test,
            y_pred,
            target_names=class_names,
            digits=4,
            zero_division=0
        )
    )

    print("Confusion Matrix:")
    cm = confusion_matrix(y_test, y_pred)

    cm_df = pd.DataFrame(
        cm,
        index=[f"True {c}" for c in class_names],
        columns=[f"Pred {c}" for c in class_names]
    )

    display(cm_df)


Decision Tree

Classification Report:
              precision    recall  f1-score   support

        Good     0.8088    0.8871    0.8462        62
        Poor     0.7955    0.7778    0.7865        45
   Resistant     0.9573    0.9345    0.9458       168
      Severe     0.9697    0.9412    0.9552        34

    accuracy                         0.9029       309
   macro avg     0.8828    0.8851    0.8834       309
weighted avg     0.9053    0.9029    0.9036       309

Confusion Matrix:


,Pred Good,Pred Poor,Pred Resistant,Pred Severe
True Good,55,4,3,0
True Poor,5,35,4,1
True Resistant,8,3,157,0
True Severe,0,2,0,32



Random Forest

Classification Report:
              precision    recall  f1-score   support

        Good     0.6087    0.6774    0.6412        62
        Poor     0.5405    0.4444    0.4878        45
   Resistant     0.8294    0.8393    0.8343       168
      Severe     0.9394    0.9118    0.9254        34

    accuracy                         0.7573       309
   macro avg     0.7295    0.7182    0.7222       309
weighted avg     0.7552    0.7573    0.7551       309

Confusion Matrix:


,Pred Good,Pred Poor,Pred Resistant,Pred Severe
True Good,42,5,14,1
True Poor,9,20,15,1
True Resistant,15,12,141,0
True Severe,3,0,0,31



Bagging

Classification Report:
              precision    recall  f1-score   support

        Good     0.8636    0.9194    0.8906        62
        Poor     0.9024    0.8222    0.8605        45
   Resistant     0.9583    0.9583    0.9583       168
      Severe     0.9706    0.9706    0.9706        34

    accuracy                         0.9320       309
   macro avg     0.9237    0.9176    0.9200       309
weighted avg     0.9325    0.9320    0.9318       309

Confusion Matrix:


,Pred Good,Pred Poor,Pred Resistant,Pred Severe
True Good,57,2,3,0
True Poor,3,37,4,1
True Resistant,6,1,161,0
True Severe,0,1,0,33



XGBoost

Classification Report:
              precision    recall  f1-score   support

        Good     0.8769    0.9194    0.8976        62
        Poor     0.9737    0.8222    0.8916        45
   Resistant     0.9593    0.9821    0.9706       168
      Severe     0.9706    0.9706    0.9706        34

    accuracy                         0.9450       309
   macro avg     0.9451    0.9236    0.9326       309
weighted avg     0.9461    0.9450    0.9444       309

Confusion Matrix:


,Pred Good,Pred Poor,Pred Resistant,Pred Severe
True Good,57,1,4,0
True Poor,4,37,3,1
True Resistant,3,0,165,0
True Severe,1,0,0,33


In [5]:
# ============================================================
# STEP 5A: XGBOOST OPTIMIZATION
# Dataset No. 2
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier


# ------------------------------------------------------------
# 1. Stratified cross-validation
# ------------------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)


# ------------------------------------------------------------
# 2. Pipeline
# Feature selection occurs INSIDE cross-validation
# ------------------------------------------------------------

xgb_pipeline = Pipeline([
    (
        "select",
        SelectKBest(
            score_func=f_classif
        )
    ),

    (
        "model",
        XGBClassifier(
            objective="multi:softprob",
            num_class=4,
            eval_metric="mlogloss",
            random_state=RANDOM_STATE,
            n_jobs=1,
            tree_method="hist"
        )
    )
])


# ------------------------------------------------------------
# 3. Hyperparameter search space
# ------------------------------------------------------------

xgb_param_grid = {

    # Dataset No. 2 contains 18 candidate features
    "select__k": [5, 10, 15, "all"],

    # Number of boosting trees
    "model__n_estimators": [100, 200, 400],

    # Tree complexity
    "model__max_depth": [3, 5, 7],

    # Shrinkage
    "model__learning_rate": [0.03, 0.05, 0.1],

    # Row sampling
    "model__subsample": [0.8, 1.0],

    # Feature sampling
    "model__colsample_bytree": [0.8, 1.0]
}


# ------------------------------------------------------------
# 4. Grid search
# ------------------------------------------------------------

xgb_grid = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=xgb_param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

print("Starting XGBoost optimization...")

start = time.time()

xgb_grid.fit(X_train, y_train)

runtime = time.time() - start


# ------------------------------------------------------------
# 5. Best CV configuration
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BEST XGBOOST CONFIGURATION")
print("=" * 70)

print("\nBest parameters:")

for parameter, value in xgb_grid.best_params_.items():
    print(f"{parameter}: {value}")

print(
    f"\nBest 5-fold CV Macro F1: "
    f"{xgb_grid.best_score_:.4f}"
)

print(
    f"Optimization runtime: "
    f"{runtime / 60:.2f} minutes"
)


# ------------------------------------------------------------
# 6. Final evaluation on LOCKED test set
# ------------------------------------------------------------

best_xgb = xgb_grid.best_estimator_

y_train_pred = best_xgb.predict(X_train)
y_test_pred = best_xgb.predict(X_test)

y_test_proba = best_xgb.predict_proba(X_test)

train_acc = accuracy_score(
    y_train,
    y_train_pred
)

test_acc = accuracy_score(
    y_test,
    y_test_pred
)

gap = train_acc - test_acc

macro_precision = precision_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

macro_auc = roc_auc_score(
    y_test,
    y_test_proba,
    multi_class="ovr",
    average="macro"
)


# ------------------------------------------------------------
# 7. Print performance
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL XGBOOST TEST PERFORMANCE")
print("=" * 70)

print(f"Train Accuracy : {train_acc:.4f}")
print(f"Test Accuracy  : {test_acc:.4f}")
print(f"Train-Test Gap : {gap:.4f}")
print(f"Macro Precision: {macro_precision:.4f}")
print(f"Macro Recall   : {macro_recall:.4f}")
print(f"Macro F1       : {macro_f1:.4f}")
print(f"Macro ROC-AUC  : {macro_auc:.4f}")


# ------------------------------------------------------------
# 8. Classification report
# ------------------------------------------------------------

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_test_pred,
        target_names=label_encoder.classes_,
        digits=4,
        zero_division=0
    )
)


# ------------------------------------------------------------
# 9. Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_test,
    y_test_pred
)

cm_df = pd.DataFrame(
    cm,
    index=[
        f"True {c}"
        for c in label_encoder.classes_
    ],
    columns=[
        f"Pred {c}"
        for c in label_encoder.classes_
    ]
)

display(cm_df)

Starting XGBoost optimization...
Fitting 5 folds for each of 432 candidates, totalling 2160 fits

BEST XGBOOST CONFIGURATION

Best parameters:
model__colsample_bytree: 0.8
model__learning_rate: 0.1
model__max_depth: 7
model__n_estimators: 400
model__subsample: 1.0
select__k: all

Best 5-fold CV Macro F1: 0.9406
Optimization runtime: 7.27 minutes

FINAL XGBOOST TEST PERFORMANCE
Train Accuracy : 1.0000
Test Accuracy  : 0.9547
Train-Test Gap : 0.0453
Macro Precision: 0.9539
Macro Recall   : 0.9380
Macro F1       : 0.9447
Macro ROC-AUC  : 0.9960

Classification Report:

              precision    recall  f1-score   support

        Good     0.9048    0.9194    0.9120        62
        Poor     0.9744    0.8444    0.9048        45
   Resistant     0.9651    0.9881    0.9765       168
      Severe     0.9714    1.0000    0.9855        34

    accuracy                         0.9547       309
   macro avg     0.9539    0.9380    0.9447       309
weighted avg     0.9550    0.9547    0.9541    

,Pred Good,Pred Poor,Pred Resistant,Pred Severe
True Good,57,1,4,0
True Poor,4,38,2,1
True Resistant,2,0,166,0
True Severe,0,0,0,34


In [6]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()  # repository root
(ROOT / "results" / "figures").mkdir(parents=True, exist_ok=True)
# ============================================================
# Export test-set predictions for Figure 4.
# Paste as the last cell of each notebook (after the optimized
# XGBoost model has been fitted). It uses the notebook's
# best_xgb, label_encoder, X_test and y_test.
# ============================================================
import pandas as pd

DATASET = 2          # set to 2 in the Dataset No. 2 notebook

proba = best_xgb.predict_proba(X_test)
classes = list(label_encoder.classes_)          # Good, Poor, Resistant, Severe
roc = pd.DataFrame(proba, columns=[f"prob_{c}" for c in classes])
roc.insert(0, "true_class", label_encoder.inverse_transform(y_test))
roc.to_csv(ROOT / "results" / f"roc_dataset{DATASET}.csv", index=False)
print(f"Saved roc_dataset{DATASET}.csv with {len(roc)} test samples")

Saved roc_dataset2.csv with 309 test samples


In [ ]:
# ============================================================
# STEP 5B: RANDOM FOREST OPTIMIZATION
# Dataset No. 2
# ============================================================

import time
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. Pipeline
# ------------------------------------------------------------

rf_pipeline = Pipeline([
    ("select", SelectKBest(
        score_func=f_classif
    )),

    ("model", RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=1
    ))
])

# ------------------------------------------------------------
# 2. Hyperparameter grid
# ------------------------------------------------------------

rf_param_grid = {

    # Dataset No. 2 contains 18 features
    "select__k": [5, 10, 15, "all"],

    # Number of trees
    "model__n_estimators": [200, 400, 600],

    # Tree depth
    "model__max_depth": [None, 10, 20, 30],

    # Minimum samples required to split
    "model__min_samples_split": [2, 5],

    # Minimum samples in leaf
    "model__min_samples_leaf": [1, 2],

    # Number of predictors considered per split
    "model__max_features": ["sqrt", 0.7, 1.0]
}

# ------------------------------------------------------------
# 3. Grid search
# ------------------------------------------------------------

rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

print("Starting Random Forest optimization...")

start = time.time()

rf_grid.fit(X_train, y_train)

runtime = time.time() - start

# ------------------------------------------------------------
# 4. Best CV configuration
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BEST RANDOM FOREST CONFIGURATION")
print("=" * 70)

print("\nBest parameters:")

for parameter, value in rf_grid.best_params_.items():
    print(f"{parameter}: {value}")

print(
    f"\nBest 5-fold CV Macro F1: "
    f"{rf_grid.best_score_:.4f}"
)

print(
    f"Optimization runtime: "
    f"{runtime / 60:.2f} minutes"
)

# ------------------------------------------------------------
# 5. Evaluate locked test set
# ------------------------------------------------------------

best_rf = rf_grid.best_estimator_

y_train_pred = best_rf.predict(X_train)
y_test_pred = best_rf.predict(X_test)

y_test_proba = best_rf.predict_proba(X_test)

train_acc = accuracy_score(
    y_train,
    y_train_pred
)

test_acc = accuracy_score(
    y_test,
    y_test_pred
)

gap = train_acc - test_acc

macro_precision = precision_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

macro_auc = roc_auc_score(
    y_test,
    y_test_proba,
    multi_class="ovr",
    average="macro"
)

print("\n" + "=" * 70)
print("FINAL RANDOM FOREST TEST PERFORMANCE")
print("=" * 70)

print(f"Train Accuracy : {train_acc:.4f}")
print(f"Test Accuracy  : {test_acc:.4f}")
print(f"Train-Test Gap : {gap:.4f}")
print(f"Macro Precision: {macro_precision:.4f}")
print(f"Macro Recall   : {macro_recall:.4f}")
print(f"Macro F1       : {macro_f1:.4f}")
print(f"Macro ROC-AUC  : {macro_auc:.4f}")

# ------------------------------------------------------------
# 6. Classification report
# ------------------------------------------------------------

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_test_pred,
        target_names=label_encoder.classes_,
        digits=4,
        zero_division=0
    )
)

# ------------------------------------------------------------
# 7. Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_test,
    y_test_pred
)

cm_df = pd.DataFrame(
    cm,
    index=[f"True {c}" for c in label_encoder.classes_],
    columns=[f"Pred {c}" for c in label_encoder.classes_]
)

display(cm_df)

Starting Random Forest optimization...
Fitting 5 folds for each of 576 candidates, totalling 2880 fits


In [ ]:
# ============================================================
# STEP 5C: BAGGING OPTIMIZATION
# Dataset No. 2
# ============================================================

import time
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. Base Decision Tree
# ------------------------------------------------------------

base_tree = DecisionTreeClassifier(
    random_state=RANDOM_STATE
)

# ------------------------------------------------------------
# 2. Pipeline
# ------------------------------------------------------------

bag_pipeline = Pipeline([
    ("select", SelectKBest(
        score_func=f_classif
    )),

    ("model", BaggingClassifier(
        estimator=base_tree,
        random_state=RANDOM_STATE,
        n_jobs=1
    ))
])

# ------------------------------------------------------------
# 3. Hyperparameter grid
# ------------------------------------------------------------

bag_param_grid = {

    # Dataset No. 2 contains 18 features
    "select__k": [5, 10, 15, "all"],

    # Number of base trees
    "model__n_estimators": [50, 100, 200],

    # Fraction of training samples for each tree
    "model__max_samples": [0.7, 0.85, 1.0],

    # Fraction of features available to each tree
    "model__max_features": [0.7, 0.85, 1.0],

    # Tree complexity
    "model__estimator__max_depth": [None, 10, 20],

    # Leaf regularization
    "model__estimator__min_samples_leaf": [1, 2]
}

# ------------------------------------------------------------
# 4. Grid search
# ------------------------------------------------------------

bag_grid = GridSearchCV(
    estimator=bag_pipeline,
    param_grid=bag_param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

print("Starting Bagging optimization...")

start = time.time()

bag_grid.fit(X_train, y_train)

runtime = time.time() - start

# ------------------------------------------------------------
# 5. Best CV configuration
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BEST BAGGING CONFIGURATION")
print("=" * 70)

print("\nBest parameters:")

for parameter, value in bag_grid.best_params_.items():
    print(f"{parameter}: {value}")

print(
    f"\nBest 5-fold CV Macro F1: "
    f"{bag_grid.best_score_:.4f}"
)

print(
    f"Optimization runtime: "
    f"{runtime / 60:.2f} minutes"
)

# ------------------------------------------------------------
# 6. Locked test evaluation
# ------------------------------------------------------------

best_bag = bag_grid.best_estimator_

y_train_pred = best_bag.predict(X_train)
y_test_pred = best_bag.predict(X_test)

y_test_proba = best_bag.predict_proba(X_test)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

gap = train_acc - test_acc

macro_precision = precision_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

macro_auc = roc_auc_score(
    y_test,
    y_test_proba,
    multi_class="ovr",
    average="macro"
)

print("\n" + "=" * 70)
print("FINAL BAGGING TEST PERFORMANCE")
print("=" * 70)

print(f"Train Accuracy : {train_acc:.4f}")
print(f"Test Accuracy  : {test_acc:.4f}")
print(f"Train-Test Gap : {gap:.4f}")
print(f"Macro Precision: {macro_precision:.4f}")
print(f"Macro Recall   : {macro_recall:.4f}")
print(f"Macro F1       : {macro_f1:.4f}")
print(f"Macro ROC-AUC  : {macro_auc:.4f}")

# ------------------------------------------------------------
# 7. Classification report
# ------------------------------------------------------------

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_test_pred,
        target_names=label_encoder.classes_,
        digits=4,
        zero_division=0
    )
)

# ------------------------------------------------------------
# 8. Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_test,
    y_test_pred
)

cm_df = pd.DataFrame(
    cm,
    index=[f"True {c}" for c in label_encoder.classes_],
    columns=[f"Pred {c}" for c in label_encoder.classes_]
)

display(cm_df)

In [ ]:
# ============================================================
# STEP 5D: DECISION TREE OPTIMIZATION
# Dataset No. 2
# ============================================================

import time
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. Pipeline
# ------------------------------------------------------------

dt_pipeline = Pipeline([
    ("select", SelectKBest(
        score_func=f_classif
    )),

    ("model", DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ))
])

# ------------------------------------------------------------
# 2. Hyperparameter grid
# ------------------------------------------------------------

dt_param_grid = {

    # Dataset No. 2 contains 18 features
    "select__k": [5, 10, 15, "all"],

    # Split criterion
    "model__criterion": ["gini", "entropy"],

    # Tree complexity
    "model__max_depth": [None, 5, 10, 15, 20, 30],

    # Minimum samples required to split a node
    "model__min_samples_split": [2, 5, 10],

    # Minimum samples in a terminal leaf
    "model__min_samples_leaf": [1, 2, 4],

    # Number of predictors considered at each split
    "model__max_features": [None, "sqrt", 0.7]
}

# ------------------------------------------------------------
# 3. Grid search
# ------------------------------------------------------------

dt_grid = GridSearchCV(
    estimator=dt_pipeline,
    param_grid=dt_param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

print("Starting Decision Tree optimization...")

start = time.time()

dt_grid.fit(X_train, y_train)

runtime = time.time() - start

# ------------------------------------------------------------
# 4. Best configuration
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BEST DECISION TREE CONFIGURATION")
print("=" * 70)

print("\nBest parameters:")

for parameter, value in dt_grid.best_params_.items():
    print(f"{parameter}: {value}")

print(
    f"\nBest 5-fold CV Macro F1: "
    f"{dt_grid.best_score_:.4f}"
)

print(
    f"Optimization runtime: "
    f"{runtime / 60:.2f} minutes"
)

# ------------------------------------------------------------
# 5. Final evaluation on locked test set
# ------------------------------------------------------------

best_dt = dt_grid.best_estimator_

y_train_pred = best_dt.predict(X_train)
y_test_pred = best_dt.predict(X_test)

y_test_proba = best_dt.predict_proba(X_test)

train_acc = accuracy_score(
    y_train,
    y_train_pred
)

test_acc = accuracy_score(
    y_test,
    y_test_pred
)

gap = train_acc - test_acc

macro_precision = precision_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

macro_auc = roc_auc_score(
    y_test,
    y_test_proba,
    multi_class="ovr",
    average="macro"
)

# ------------------------------------------------------------
# 6. Print final performance
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL DECISION TREE TEST PERFORMANCE")
print("=" * 70)

print(f"Train Accuracy : {train_acc:.4f}")
print(f"Test Accuracy  : {test_acc:.4f}")
print(f"Train-Test Gap : {gap:.4f}")
print(f"Macro Precision: {macro_precision:.4f}")
print(f"Macro Recall   : {macro_recall:.4f}")
print(f"Macro F1       : {macro_f1:.4f}")
print(f"Macro ROC-AUC  : {macro_auc:.4f}")

# ------------------------------------------------------------
# 7. Classification report
# ------------------------------------------------------------

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_test_pred,
        target_names=label_encoder.classes_,
        digits=4,
        zero_division=0
    )
)

# ------------------------------------------------------------
# 8. Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_test,
    y_test_pred
)

cm_df = pd.DataFrame(
    cm,
    index=[f"True {c}" for c in label_encoder.classes_],
    columns=[f"Pred {c}" for c in label_encoder.classes_]
)

display(cm_df)

In [ ]:
# ============================================================
# STEP 6: FIVE-SEED ROBUSTNESS TEST
# Dataset No. 2
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from xgboost import XGBClassifier


# ------------------------------------------------------------
# 1. Random seeds
# ------------------------------------------------------------

SEEDS = [0, 1, 2, 3, 4]


# ------------------------------------------------------------
# 2. Fixed optimized configurations
# These are the hyperparameters selected using Dataset No. 2
# ------------------------------------------------------------

def get_optimized_models(seed):

    models = {

        # ----------------------------------------------------
        # Decision Tree
        # Best feature selection: k = 15
        # ----------------------------------------------------
        "Decision Tree": Pipeline([
            (
                "select",
                SelectKBest(
                    score_func=f_classif,
                    k=15
                )
            ),
            (
                "model",
                DecisionTreeClassifier(
                    criterion="entropy",
                    max_depth=None,
                    max_features=None,
                    min_samples_leaf=1,
                    min_samples_split=5,
                    random_state=seed
                )
            )
        ]),

        # ----------------------------------------------------
        # Random Forest
        # Best feature selection: all 17 features
        # ----------------------------------------------------
        "Random Forest": RandomForestClassifier(
            n_estimators=200,
            max_depth=None,
            max_features=1.0,
            min_samples_leaf=1,
            min_samples_split=5,
            random_state=seed,
            n_jobs=-1
        ),

        # ----------------------------------------------------
        # Bagging
        # Best feature selection: k = 15
        # ----------------------------------------------------
        "Bagging": Pipeline([
            (
                "select",
                SelectKBest(
                    score_func=f_classif,
                    k=15
                )
            ),
            (
                "model",
                BaggingClassifier(
                    estimator=DecisionTreeClassifier(
                        max_depth=None,
                        min_samples_leaf=1,
                        random_state=seed
                    ),
                    n_estimators=200,
                    max_samples=1.0,
                    max_features=0.85,
                    random_state=seed,
                    n_jobs=-1
                )
            )
        ]),

        # ----------------------------------------------------
        # XGBoost
        # Best feature selection: all 17 features
        # ----------------------------------------------------
        "XGBoost": XGBClassifier(
            objective="multi:softprob",
            num_class=4,
            eval_metric="mlogloss",

            n_estimators=400,
            max_depth=7,
            learning_rate=0.1,
            subsample=1.0,
            colsample_bytree=0.8,

            tree_method="hist",
            random_state=seed,
            n_jobs=-1
        )
    }

    return models


# ------------------------------------------------------------
# 3. Run five-seed experiment
# ------------------------------------------------------------

robustness_results = []

for seed in SEEDS:

    print("\n" + "=" * 80)
    print(f"RANDOM SEED: {seed}")
    print("=" * 80)

    # --------------------------------------------------------
    # New stratified 80:20 split for each seed
    # --------------------------------------------------------

    X_train_seed, X_test_seed, y_train_seed, y_test_seed = (
        train_test_split(
            X,
            y,
            test_size=0.20,
            random_state=seed,
            stratify=y
        )
    )

    models = get_optimized_models(seed)

    # --------------------------------------------------------
    # Train each model
    # --------------------------------------------------------

    for name, model in models.items():

        start = time.time()

        model.fit(
            X_train_seed,
            y_train_seed
        )

        runtime = time.time() - start

        # Predictions
        train_pred = model.predict(X_train_seed)
        test_pred = model.predict(X_test_seed)

        # Probabilities for ROC-AUC
        test_proba = model.predict_proba(X_test_seed)

        # ----------------------------------------------------
        # Metrics
        # ----------------------------------------------------

        train_acc = accuracy_score(
            y_train_seed,
            train_pred
        )

        test_acc = accuracy_score(
            y_test_seed,
            test_pred
        )

        gap = train_acc - test_acc

        macro_precision = precision_score(
            y_test_seed,
            test_pred,
            average="macro",
            zero_division=0
        )

        macro_recall = recall_score(
            y_test_seed,
            test_pred,
            average="macro",
            zero_division=0
        )

        macro_f1 = f1_score(
            y_test_seed,
            test_pred,
            average="macro",
            zero_division=0
        )

        macro_auc = roc_auc_score(
            y_test_seed,
            test_proba,
            multi_class="ovr",
            average="macro"
        )

        # ----------------------------------------------------
        # Save results
        # ----------------------------------------------------

        robustness_results.append({

            "Seed": seed,
            "Model": name,

            "Train Accuracy": train_acc,
            "Test Accuracy": test_acc,
            "Gap": gap,

            "Macro Precision": macro_precision,
            "Macro Recall": macro_recall,
            "Macro F1": macro_f1,
            "Macro ROC-AUC": macro_auc,

            "Runtime (s)": runtime
        })

        print(
            f"{name:<15} | "
            f"Train={train_acc:.4f} | "
            f"Test={test_acc:.4f} | "
            f"F1={macro_f1:.4f} | "
            f"AUC={macro_auc:.4f} | "
            f"Gap={gap:.4f}"
        )


# ------------------------------------------------------------
# 4. Full seed-by-seed results
# ------------------------------------------------------------

robustness_df = pd.DataFrame(
    robustness_results
)

print("\n" + "=" * 100)
print("RESULTS FOR EVERY RANDOM SEED - DATASET NO. 2")
print("=" * 100)

display(robustness_df)

In [ ]:
# ============================================================
# STEP 6B: FIVE-SEED ROBUSTNESS SUMMARY
# Dataset No. 2
# ============================================================

metric_columns = [
    "Train Accuracy",
    "Test Accuracy",
    "Gap",
    "Macro Precision",
    "Macro Recall",
    "Macro F1",
    "Macro ROC-AUC"
]

# Calculate mean and standard deviation
summary_mean = (
    robustness_df
    .groupby("Model")[metric_columns]
    .mean()
)

summary_std = (
    robustness_df
    .groupby("Model")[metric_columns]
    .std()
)


# ------------------------------------------------------------
# Format as mean ± standard deviation
# ------------------------------------------------------------

summary_rows = []

for model in summary_mean.index:

    row = {
        "Model": model
    }

    for metric in metric_columns:

        mean_value = summary_mean.loc[
            model,
            metric
        ] * 100

        std_value = summary_std.loc[
            model,
            metric
        ] * 100

        row[metric] = (
            f"{mean_value:.2f} ± "
            f"{std_value:.2f}"
        )

    summary_rows.append(row)


robustness_summary_df = pd.DataFrame(
    summary_rows
)


# ------------------------------------------------------------
# Rename columns for publication
# ------------------------------------------------------------

robustness_summary_df = (
    robustness_summary_df.rename(
        columns={
            "Train Accuracy":
                "Train Accuracy (%)",

            "Test Accuracy":
                "Test Accuracy (%)",

            "Gap":
                "Generalization Gap (pp)",

            "Macro Precision":
                "Macro Precision (%)",

            "Macro Recall":
                "Macro Recall (%)",

            "Macro F1":
                "Macro F1 (%)",

            "Macro ROC-AUC":
                "Macro ROC-AUC (%)"
        }
    )
)


# ------------------------------------------------------------
# Sort by mean test accuracy
# ------------------------------------------------------------

model_order = (
    robustness_df
    .groupby("Model")["Test Accuracy"]
    .mean()
    .sort_values(ascending=False)
    .index
)

robustness_summary_df["Model"] = pd.Categorical(
    robustness_summary_df["Model"],
    categories=model_order,
    ordered=True
)

robustness_summary_df = (
    robustness_summary_df
    .sort_values("Model")
    .reset_index(drop=True)
)


print("\n" + "=" * 105)
print(
    "FIVE-SEED ROBUSTNESS SUMMARY - "
    "DATASET NO. 2"
)
print("=" * 105)

display(robustness_summary_df)

In [ ]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()  # repository root
(ROOT / "results" / "figures").mkdir(parents=True, exist_ok=True)
# ============================================================
# FIGURE 3b
# Train-test accuracy comparison
# Dataset No. 2
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Model names
# ------------------------------------------------------------

models = [
    "Decision Tree",
    "Random Forest",
    "Bagging",
    "XGBoost"
]

# ------------------------------------------------------------
# 2. Baseline results
# Final results after removing constant Temperature feature
# ------------------------------------------------------------

baseline_train = [
    100.00,   # Decision Tree
    100.00,   # Random Forest
    100.00,   # Bagging
    100.00    # XGBoost
]

baseline_test = [
    90.29,    # Decision Tree
    75.73,    # Random Forest
    93.20,    # Bagging
    94.50     # XGBoost
]

# ------------------------------------------------------------
# 3. Optimized results
# Locked test-set results from Step 5
# ------------------------------------------------------------

optimized_train = [
    98.06,    # Decision Tree
    99.27,    # Random Forest
    100.00,   # Bagging
    100.00    # XGBoost
]

optimized_test = [
    91.91,    # Decision Tree
    94.17,    # Random Forest
    92.88,    # Bagging
    95.47     # XGBoost
]

# ------------------------------------------------------------
# 4. Bar positions
# ------------------------------------------------------------

x = np.arange(len(models))
width = 0.19

# ------------------------------------------------------------
# 5. Create figure
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 6))

bars1 = ax.bar(
    x - 1.5 * width,
    baseline_train,
    width,
    label="Baseline Train",
    color="#9ecae1"
)

bars2 = ax.bar(
    x - 0.5 * width,
    baseline_test,
    width,
    label="Baseline Test",
    color="#3182bd"
)

bars3 = ax.bar(
    x + 0.5 * width,
    optimized_train,
    width,
    label="Optimized Train",
    color="#a1d99b"
)

bars4 = ax.bar(
    x + 1.5 * width,
    optimized_test,
    width,
    label="Optimized Test",
    color="#31a354"
)

# ------------------------------------------------------------
# 6. Labels and formatting
# ------------------------------------------------------------

ax.set_ylabel("Accuracy (%)")
ax.set_xlabel("Machine Learning Model")

ax.set_title("(b) Dataset No. 2")

ax.set_xticks(x)
ax.set_xticklabels(models)

# Keep same scale as Figure 3a
ax.set_ylim(0, 120)

ax.legend(ncol=2)

# ------------------------------------------------------------
# 7. Add numerical values above bars
# ------------------------------------------------------------

for bars in [bars1, bars2, bars3, bars4]:

    ax.bar_label(
        bars,
        fmt="%.2f",
        padding=2,
        fontsize=8
    )

# ------------------------------------------------------------
# 8. Final layout
# ------------------------------------------------------------

plt.tight_layout()

# ------------------------------------------------------------
# 9. Save publication-quality figure
# ------------------------------------------------------------

plt.savefig(
    ROOT / "results" / "figures" / "Figure3b_Dataset2_Accuracy.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()  # repository root
(ROOT / "results" / "figures").mkdir(parents=True, exist_ok=True)
# ============================================================
# FIGURE 3b
# Train-test accuracy comparison with robustness error bars
# Dataset No. 2
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Model names
# ------------------------------------------------------------

models = [
    "Decision Tree",
    "Random Forest",
    "Bagging",
    "XGBoost"
]

# ------------------------------------------------------------
# 2. Baseline results
# Held-out train-test split
# ------------------------------------------------------------

baseline_train = [
    100.00,   # Decision Tree
    100.00,   # Random Forest
    100.00,   # Bagging
    100.00    # XGBoost
]

baseline_test = [
    90.29,    # Decision Tree
    75.73,    # Random Forest
    93.20,    # Bagging
    94.50     # XGBoost
]

# ------------------------------------------------------------
# 3. Optimized results
# Held-out train-test split from Step 5
# ------------------------------------------------------------

optimized_train = [
    98.06,    # Decision Tree
    99.27,    # Random Forest
    100.00,   # Bagging
    100.00    # XGBoost
]

optimized_test = [
    91.91,    # Decision Tree
    94.17,    # Random Forest
    92.88,    # Bagging
    95.47     # XGBoost
]

# ------------------------------------------------------------
# 4. Standard deviations from five-seed robustness test
# ------------------------------------------------------------

optimized_train_std = [
    0.19,     # Decision Tree
    0.21,     # Random Forest
    0.00,     # Bagging
    0.00      # XGBoost
]

optimized_test_std = [
    1.52,     # Decision Tree
    1.13,     # Random Forest
    0.84,     # Bagging
    0.83      # XGBoost
]

# ------------------------------------------------------------
# 5. Bar positions
# ------------------------------------------------------------

x = np.arange(len(models))
width = 0.19

# ------------------------------------------------------------
# 6. Create larger figure
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 8))

bars1 = ax.bar(
    x - 1.5 * width,
    baseline_train,
    width,
    label="Baseline Train",
    color="#9ecae1"
)

bars2 = ax.bar(
    x - 0.5 * width,
    baseline_test,
    width,
    label="Baseline Test",
    color="#3182bd"
)

bars3 = ax.bar(
    x + 0.5 * width,
    optimized_train,
    width,
    yerr=optimized_train_std,
    capsize=5,
    label="Optimized Train",
    color="#a1d99b"
)

bars4 = ax.bar(
    x + 1.5 * width,
    optimized_test,
    width,
    yerr=optimized_test_std,
    capsize=5,
    label="Optimized Test",
    color="#31a354"
)

# ------------------------------------------------------------
# 7. Labels and formatting
# ------------------------------------------------------------

ax.set_ylabel(
    "Accuracy (%)",
    fontsize=20,
)

ax.set_xlabel(
    "Machine Learning Model",
    fontsize=20,
)

ax.set_title(
    "(b) Dataset No. 2",
    fontsize=18,
    pad=15
)

ax.set_xticks(x)

ax.set_xticklabels(
    models,
    fontsize=16
)

ax.tick_params(
    axis="y",
    labelsize=16
)

# Same scale as Dataset No. 1
ax.set_ylim(0, 120)

ax.legend(
    ncol=2,
    fontsize=16,
    frameon=True
)

# ------------------------------------------------------------
# 8. Numerical values above bars
# ------------------------------------------------------------

for bars in [bars1, bars2]:
    ax.bar_label(
        bars,
        fmt="%.2f",
        padding=3,
        fontsize=11
    )

# Extra padding because of error bars
for bars in [bars3, bars4]:
    ax.bar_label(
        bars,
        fmt="%.2f",
        padding=8,
        fontsize=11
    )

# ------------------------------------------------------------
# 9. Final layout
# ------------------------------------------------------------

plt.tight_layout()

# ------------------------------------------------------------
# 10. Save
# ------------------------------------------------------------

plt.savefig(
    ROOT / "results" / "figures" / "Figure3b_Dataset2_Accuracy.png",
    dpi=800,
    bbox_inches="tight"
)

plt.show()

In [ ]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()  # repository root
(ROOT / "results" / "figures").mkdir(parents=True, exist_ok=True)
# ============================================================
# FIGURE 4b
# ROC-AUC curves for optimized XGBoost
# Dataset No. 2
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

# ------------------------------------------------------------
# 1. Predict probabilities
# ------------------------------------------------------------

y_score = best_xgb.predict_proba(X_test)

classes = np.arange(len(label_encoder.classes_))

y_test_bin = label_binarize(
    y_test,
    classes=classes
)

# ------------------------------------------------------------
# 2. ROC curves
# ------------------------------------------------------------

plt.figure(figsize=(10, 8))

auc_values = []

for i, class_name in enumerate(label_encoder.classes_):

    fpr, tpr, _ = roc_curve(
        y_test_bin[:, i],
        y_score[:, i]
    )

    class_auc = auc(
        fpr,
        tpr
    )

    auc_values.append(class_auc)

    plt.plot(
        fpr,
        tpr,
        linewidth=2.5,
        label=f"{class_name} (AUC = {class_auc:.3f})"
    )

# ------------------------------------------------------------
# 3. Random classifier reference
# ------------------------------------------------------------

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=2,
    label="Random classifier"
)

macro_auc = np.mean(auc_values)

# ------------------------------------------------------------
# 4. Labels and formatting
# ------------------------------------------------------------

plt.xlabel(
    "False Positive Rate",
    fontsize=18,
)

plt.ylabel(
    "True Positive Rate",
    fontsize=18,
)

plt.title(
    f"(b) Dataset No. 2\n"
    f"Optimized XGBoost, Macro AUC = {macro_auc:.3f}",
    fontsize=16,
    pad=15
)

plt.xticks(fontsize=16)
plt.yticks(fontsize=16)

plt.legend(
    loc="lower right",
    fontsize=18
)

plt.grid()

plt.xlim([0, 1])
plt.ylim([0, 1.02])

# ------------------------------------------------------------
# 5. Final layout and save
# ------------------------------------------------------------

plt.tight_layout()

plt.savefig(
    ROOT / "results" / "figures" / "Figure4b_Dataset2_XGBoost_ROC.png",
    dpi=800,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 6. Print numerical values
# ------------------------------------------------------------

print("Class-specific ROC-AUC:")

for name, score in zip(
    label_encoder.classes_,
    auc_values
):
    print(f"{name:<12}: {score:.4f}")

print(f"\nMacro ROC-AUC: {macro_auc:.4f}")

In [ ]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()  # repository root
(ROOT / "results" / "figures").mkdir(parents=True, exist_ok=True)
# ============================================================
# FIGURE 5b
# XGBoost Feature Importance
# Dataset No. 2
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Extract trained XGBoost model from pipeline
# ------------------------------------------------------------

xgb_model = best_xgb.named_steps["model"]

# ------------------------------------------------------------
# 2. Feature names
# ------------------------------------------------------------

# SelectKBest selected "all", so all remaining X columns
# correspond directly to the trained XGBoost model.

feature_names = X.columns

# ------------------------------------------------------------
# 3. Create importance table
# ------------------------------------------------------------

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": xgb_model.feature_importances_
})

# Sort highest to lowest
importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

print("=" * 70)
print("XGBOOST FEATURE IMPORTANCE - DATASET NO. 2")
print("=" * 70)

display(importance_df)

# ------------------------------------------------------------
# 4. Top 10 features
# ------------------------------------------------------------

top_n = 10

top_features = (
    importance_df
    .head(top_n)
    .sort_values(
        "Importance",
        ascending=True
    )
)

# ------------------------------------------------------------
# 5. Larger plot
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 8))

ax.barh(
    top_features["Feature"],
    top_features["Importance"]
)

ax.set_xlabel(
    "Feature Importance",
    fontsize=20,
)

ax.set_ylabel(
    "Feature",
    fontsize=20,
)

ax.set_title(
    "(b) Dataset No. 2",
    fontsize=18,
    pad=15
)

ax.tick_params(
    axis="x",
    labelsize=18
)

ax.tick_params(
    axis="y",
    labelsize=18
)

# ------------------------------------------------------------
# 6. Final layout
# ------------------------------------------------------------

plt.tight_layout()

# ------------------------------------------------------------
# 7. Save
# ------------------------------------------------------------

plt.savefig(
    ROOT / "results" / "figures" / "Figure5b_Dataset2_FeatureImportance.png",
    dpi=800,
    bbox_inches="tight"
)

plt.show()

In [ ]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()  # repository root
(ROOT / "results" / "figures").mkdir(parents=True, exist_ok=True)
# ============================================================
# REDUCED-FEATURE XGBOOST EXPERIMENT: DATASET NO. 2
# (supervisor comment 2)
#
# Standalone script. Reproduces the notebook preprocessing and
# the optimized XGBoost hyperparameters, then retrains XGBoost
# using only the top-k descriptors of its own feature-importance
# ranking and compares with the full descriptor set.
#
# No test-set leakage: the importance ranking is always computed
# from an XGBoost model fitted on the TRAINING subset only.
#
# Dataset No. 2 only. Place this script in the same folder
# as "Dataset No.2.csv" (in the data/ folder of the repository).
#
# Usage:
#   python reduced_feature_xgboost_dataset2.py
#
# Outputs (reduced_feature_results/):
#   dataset{N}_locked_test.csv      random state 42 (as Table 3)
#   dataset{N}_five_seed_raw.csv    seeds 0-4 (as Table 4)
#   dataset{N}_five_seed_summary.csv
#   dataset{N}_top_features.csv     top-k lists used
# ============================================================

import os

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier


# ============================================================
# 1. SETTINGS
# ============================================================

DATASET = 2

CSV_PATH = ROOT / "data" / "Dataset No.2.csv"

# Optimized hyperparameters from the notebooks (grid search,
# Step 5A). Both selected select__k = "all".
XGB_PARAMS = {
    2: dict(n_estimators=400, max_depth=7, learning_rate=0.1,
            subsample=1.0, colsample_bytree=0.8),
}

# Values reported in the manuscript, used as a sanity check
# for the full-descriptor ("all") runs.
EXPECTED = {
    2: {"locked_test_acc": 95.47, "five_seed_test_acc": 96.44},
}

TOP_K_VALUES = [5, 10, 15]     # top-10 corresponds to Figure 5
LOCKED_STATE = 42
SEEDS = [0, 1, 2, 3, 4]
TEST_SIZE = 0.20

OUT_DIR = ROOT / "results" / "reduced_feature_results"
os.makedirs(OUT_DIR, exist_ok=True)


# ============================================================
# 2. LOAD AND PREPROCESS (identical to notebook Steps 1-2)
# ============================================================

df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()

label_encoder = LabelEncoder()
y = pd.Series(
    label_encoder.fit_transform(df["Behavior"]),
    index=df.index,
    name="Behavior",
)

X = df.drop(columns=["ASTM Material", "Behavior", "Pitting_label"]).copy()

# Remove zero-variance features (temperature) and clean
# feature names for XGBoost, as in the Dataset No. 2 notebook.
constant_features = [c for c in X.columns if X[c].nunique() <= 1]
X = X.drop(columns=constant_features)
X.columns = (
    X.columns.str.strip()
    .str.replace("[", "", regex=False)
    .str.replace("]", "", regex=False)
    .str.replace("<", "", regex=False)
)

non_numeric = X.select_dtypes(exclude=np.number).columns.tolist()
if non_numeric:
    raise ValueError(f"Non-numeric features found: {non_numeric}")

print("=" * 70)
print(f"REDUCED-FEATURE XGBOOST | DATASET NO. {DATASET}")
print("=" * 70)
print(f"Samples: {len(X)} | Candidate descriptors: {X.shape[1]}")
print("Class encoding:", dict(enumerate(label_encoder.classes_)))

K_LIST = [k for k in TOP_K_VALUES if k < X.shape[1]] + ["all"]


# ============================================================
# 3. HELPERS
# ============================================================

def make_xgb(seed, n_jobs):
    return XGBClassifier(
        objective="multi:softprob",
        num_class=4,
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=seed,
        n_jobs=n_jobs,
        **XGB_PARAMS[DATASET],
    )


def importance_ranking(X_train, y_train, seed, n_jobs):
    """Rank descriptors with XGBoost fitted on training data only."""
    model = make_xgb(seed, n_jobs).fit(X_train, y_train)
    return (
        pd.Series(model.feature_importances_, index=X_train.columns)
        .sort_values(ascending=False)
        .index.tolist()
    )


def evaluate(X_train, X_test, y_train, y_test, seed, n_jobs):
    model = make_xgb(seed, n_jobs).fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train)) * 100
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)
    test_acc = accuracy_score(y_test, pred) * 100
    return {
        "n_features": X_train.shape[1],
        "Train Accuracy (%)": train_acc,
        "Test Accuracy (%)": test_acc,
        "Generalization Gap (pp)": train_acc - test_acc,
        "Macro F1 (%)": f1_score(y_test, pred, average="macro") * 100,
        "Macro ROC-AUC (%)": roc_auc_score(
            y_test, proba, multi_class="ovr", average="macro") * 100,
    }


# ============================================================
# 4. LOCKED TEST SET (random state 42, as in Table 3)
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=LOCKED_STATE, stratify=y,
)

# n_jobs=1 matches the notebook grid-search model.
ranking = importance_ranking(X_train, y_train, LOCKED_STATE, n_jobs=1)

rows, top_lists = [], []
for k in K_LIST:
    cols = list(X_train.columns) if k == "all" else ranking[:k]
    res = evaluate(X_train[cols], X_test[cols], y_train, y_test,
                   LOCKED_STATE, n_jobs=1)
    rows.append({"k": k, **res})
    if k != "all":
        top_lists.append({"k": k, "features": "; ".join(cols)})

locked = pd.DataFrame(rows).round(2)
locked.to_csv(os.path.join(OUT_DIR, f"dataset{DATASET}_locked_test.csv"),
              index=False)
pd.DataFrame(top_lists).to_csv(
    os.path.join(OUT_DIR, f"dataset{DATASET}_top_features.csv"), index=False)

print("\nLOCKED TEST SET (random state 42)")
print(locked.to_string(index=False))
print("\nTop-10 descriptors (training-set ranking):")
for i, name in enumerate(ranking[:10], start=1):
    print(f"  {i:2d}. {name}")


# ============================================================
# 5. FIVE-SEED ROBUSTNESS (seeds 0-4, as in Table 4)
#    Ranking recomputed on each seed's training subset.
# ============================================================

rows = []
for seed in SEEDS:
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=seed, stratify=y,
    )
    rank_s = importance_ranking(Xtr, ytr, seed, n_jobs=-1)
    for k in K_LIST:
        cols = list(Xtr.columns) if k == "all" else rank_s[:k]
        res = evaluate(Xtr[cols], Xte[cols], ytr, yte, seed, n_jobs=-1)
        rows.append({"seed": seed, "k": k, **res})

raw = pd.DataFrame(rows)
raw.to_csv(os.path.join(OUT_DIR, f"dataset{DATASET}_five_seed_raw.csv"),
           index=False)

metrics = [c for c in raw.columns if c not in ("seed", "k", "n_features")]
summary = raw.groupby("k", sort=False)[metrics].agg(["mean", "std"]).round(2)
summary.to_csv(os.path.join(OUT_DIR, f"dataset{DATASET}_five_seed_summary.csv"))

print("\nFIVE-SEED SUMMARY (mean, std)")
print(summary.to_string())


# ============================================================
# 6. SANITY CHECK AGAINST THE MANUSCRIPT
# ============================================================

got_locked = float(locked.loc[locked["k"] == "all", "Test Accuracy (%)"].iloc[0])
got_seed = float(
    raw.loc[raw["k"] == "all", "Test Accuracy (%)"].mean().round(2)
)
exp = EXPECTED[DATASET]

print("\nSANITY CHECK (full descriptor set)")
print(f"  Locked test accuracy : {got_locked:.2f}%  "
      f"(manuscript Table 3: {exp['locked_test_acc']:.2f}%)")
print(f"  Five-seed mean       : {got_seed:.2f}%  "
      f"(manuscript Table 4: {exp['five_seed_test_acc']:.2f}%)")

if (abs(got_locked - exp["locked_test_acc"]) > 0.05
        or abs(got_seed - exp["five_seed_test_acc"]) > 0.05):
    print("  WARNING: does not reproduce the manuscript. Check the CSV "
          "file and package versions before using these results.")
else:
    print("  OK: the full-descriptor runs reproduce the manuscript.")